# **Decision Tree Classifier**

### **Dataset Generation**

In [37]:
# Toy dataset

training_data = [
    ['Green', 3, 'Grape'],
    ['Yellow', 3, 'Lemon'],
    ['Red', 1, 'Apple'],
    ['Red', 1, 'Apple'],
    ['Yellow', 3, 'Lemon'],
]

# Column labels
header = ['color', 'diameter', 'label']

### **Unique Values**

In [38]:
# find the unique values for a column in a dataset.
def unique_vals(rows, col):
  return set([row[col] for row in rows])

In [39]:
unique_vals(training_data, 0)

{'Green', 'Red', 'Yellow'}

In [40]:
unique_vals(training_data, 2)

{'Apple', 'Grape', 'Lemon'}

### **Class Counts**

In [41]:
# counts the number of each type of example in a dataset
def class_counts(rows):
  counts = {}
  for row in rows:
    label = row[-1]
    if label not in counts:
      counts[label] = 0
    counts[label] += 1
  return counts

In [42]:
class_counts(training_data)

{'Grape': 1, 'Lemon': 2, 'Apple': 2}

### **Numeric Value**

In [43]:
# test if value is numeric
def is_numeric(value):
  return isinstance(value, int) or isinstance(value, float)

In [44]:
is_numeric(7)

True

In [45]:
is_numeric(8.1)

True

### **Asking Questions**

A Question is used to partition a dataset.

    This class just records a 'column number' (e.g., 0 for Color) and a
    'column value' (e.g., Green). The 'match' method is used to compare
    the feature value in an example to the feature value stored in the
    question

In [46]:
class Question:

  def __init__(self, column, value):
    self.column = column
    self.value = value

  def match(self, example):
    # Compare the feature value in an example to the
    # feature value in this question.
    val = example[self.column]
    if is_numeric(val):
      return val >= self.value
    else:
      return val == self.value

  def __repr__(self):
    # This is just a helper method to print
    # the question in a readable format.
    condition = "=="
    if is_numeric(self.value):
      condition = ">="
    return "Is %s %s %s?" % (
        header[self.column], condition, str(self.value)
    )


In [47]:
# question for a numeric attribute
Question(1, 3)

Is diameter >= 3?

In [48]:
# categorical attribute
q = Question(0, 'Green')
q

Is color == Green?

In [49]:
example = training_data[0]
q.match(example)  # this will be true, since the first example is Green.


True

### **Partition**

Partitions a dataset.

    For each row in the dataset, check if it matches the question. If
    so, add it to 'true rows', otherwise, add it to 'false rows'.

In [50]:
def partition(rows, question):
  true_rows, false_rows = [], []
  for row in rows:
    if question.match(row):
      true_rows.append(row)
    else:
      false_rows.append(row)
  return true_rows, false_rows

In [51]:
# partition the training data based on whether rows are Red.
true_rows, false_rows = partition(training_data, Question(0, 'Red'))
true_rows

[['Red', 1, 'Apple'], ['Red', 1, 'Apple']]

In [52]:
false_rows

[['Green', 3, 'Grape'], ['Yellow', 3, 'Lemon'], ['Yellow', 3, 'Lemon']]

### **Ginni**

In [53]:
# Calculate the Gini Impurity for a list of rows.
def gini(rows):
  counts = class_counts(rows)
  impurity = 1
  for lbl in counts:
    prob_of_lbl = counts[lbl] / float(len(rows))
    impurity -= prob_of_lbl ** 2
  return impurity

In [54]:
# a dataset with no mixing.
no_mixing = [['Apple'], ['Apple']]
gini(no_mixing)

0.0

In [55]:
# dataset with a 50:50 apples:oranges ratio
some_mixing = [['Apple'], ['Orange']]
# this will return 0.5 - meaning, there's a 50% chance of misclassifying
# a random example we draw from the dataset.
gini(some_mixing)

0.5

In [56]:
# dataset with many different labels
lots_of_mixing = [['Apple'],
                  ['Orange'],
                  ['Grape'],
                  ['Grapefruit'],
                  ['Blueberry']]
# This will return 0.8
gini(lots_of_mixing)

0.7999999999999998

### **Information Gain**



    The uncertainty of the starting node, minus the weighted impurity of
    two child nodes.


In [57]:
def info_gain(left, right, current_uncertainity):
  p = float(len(left)) / (len(left) + len(right))
  return current_uncertainity - p * gini(left) - (1-p) * gini(right)

In [58]:
# uncertainy of our training data.
current_uncertainty = gini(training_data)
current_uncertainty

0.6399999999999999

In [59]:
# How much information do we gain by partioning on 'Green'?
true_rows, false_rows = partition(training_data, Question(0, 'Green'))
info_gain(true_rows, false_rows, current_uncertainty)

0.23999999999999988

In [60]:
# What about if we partioned on 'Red' instead?
true_rows, false_rows = partition(training_data, Question(0,'Red'))
info_gain(true_rows, false_rows, current_uncertainty)

0.37333333333333324

In [61]:
# we learned more using Red (0.37) than Green (0.14)
# Why?

In [63]:
true_rows, false_rows = partition(training_data, Question(0,'Red'))

# Here the true_rows contain only Apples
true_rows

[['Red', 1, 'Apple'], ['Red', 1, 'Apple']]

In [64]:
# And the false rows contain two types of fruit. Not too bad.
false_rows

[['Green', 3, 'Grape'], ['Yellow', 3, 'Lemon'], ['Yellow', 3, 'Lemon']]

In [65]:
# On the other hand, partitioning by Green doesn't help so much.
true_rows, false_rows = partition(training_data, Question(0,'Green'))

# We've isolated one apple in the true rows.
true_rows

[['Green', 3, 'Grape']]

In [67]:
# But the false-rows are badly mixed up
false_rows

[['Yellow', 3, 'Lemon'],
 ['Red', 1, 'Apple'],
 ['Red', 1, 'Apple'],
 ['Yellow', 3, 'Lemon']]

### **Best Split**

    Find the best question to ask by iterating over every feature / value
    and calculating the information gain.


In [68]:
def find_best_split(rows):
  best_gain = 0   # keep track of the best information gain
  best_question = None  # keep train of the feature / value that produced it
  n_features = len(rows[0]) - 1 # number of columns

  for col in range(n_features):

    values = set([row[col] for row in rows]) # unique values in the column

    for val in values:  # for each values

      question = Question(col, val)

      # try splitting the dataset
      true_rows, false_rows = partition(rows, question)

      # skip this split if it dosen't divide the datset
      if len(true_rows) == 0 or len(false_rows) == 0:
        continue

      # Calculate the information gain from this split
      gain = info_gain(true_rows, false_rows, current_uncertainty)

      if gain >= best_gain:
        best_gain, best_question = gain, question

  return best_gain, best_question

In [69]:
# Find the best question to ask first for our toy dataset.
best_gain, best_question = find_best_split(training_data)
best_question

Is diameter >= 3?

### **Leaf**


    A Leaf node classifies data.
    This holds a dictionary of class (e.g., "Apple") -> number of times
    it appears in the rows from the training data that reach this leaf.
  

In [74]:
class Leaf:

  def __init__(self, rows):
    self.predictions = class_counts(rows)

### **Decision Node**

  
    A Decision Node asks a question.
    This holds a reference to the question, and to the two child nodes.
  

In [70]:
class Decision_Node:

  def __init__(self,
               question,
               true_branch,
               false_branch):
    self.question = question
    self.true_branch = true_branch
    self.false_branch = false_branch

### **Build Tree**


    Rules of recursion:
    1) Believe that it works.
    2) Start by checking for the base case (no further information gain).
    3) Prepare for giant stack traces.
  

In [71]:
def build_tree(rows):

    # Try partitioing the dataset on each of the unique attribute,
    # calculate the information gain,
    # and return the question that produces the highest gain.
    gain, question = find_best_split(rows)

    # Base case: no further info gain
    # Since we can ask no further questions,
    # we'll return a leaf.
    if gain == 0:
        return Leaf(rows)

    # If we reach here, we have found a useful feature / value
    # to partition on.
    true_rows, false_rows = partition(rows, question)

    # Recursively build the true branch.
    true_branch = build_tree(true_rows)

    # Recursively build the false branch.
    false_branch = build_tree(false_rows)

    # Return a Question node.
    # This records the best feature / value to ask at this point,
    # as well as the branches to follow
    # dependingo on the answer.
    return Decision_Node(question, true_branch, false_branch)

### **Print Tree**

In [75]:
def print_tree(node, spacing=""):

    # Base case: we've reached a leaf
    if isinstance(node, Leaf):
        print (spacing + "Predict", node.predictions)
        return

    # Print the question at this node
    print (spacing + str(node.question))

    # Call this function recursively on the true branch
    print (spacing + '--> True:')
    print_tree(node.true_branch, spacing + "  ")

    # Call this function recursively on the false branch
    print (spacing + '--> False:')
    print_tree(node.false_branch, spacing + "  ")

In [76]:
my_tree = build_tree(training_data)

In [77]:
print_tree(my_tree)

Is diameter >= 3?
--> True:
  Is color == Yellow?
  --> True:
    Predict {'Lemon': 2}
  --> False:
    Predict {'Grape': 1}
--> False:
  Predict {'Apple': 2}


### **Classify**

In [78]:
def classify(row, node):

    # Base case: we've reached a leaf
    if isinstance(node, Leaf):
        return node.predictions

    # Decide whether to follow the true-branch or the false-branch.
    # Compare the feature / value stored in the node,
    # to the example we're considering.
    if node.question.match(row):
        return classify(row, node.true_branch)
    else:
        return classify(row, node.false_branch)

In [79]:
# The tree predicts the 1st row of our
# training data is an apple with confidence 1.
classify(training_data[0], my_tree)

{'Grape': 1}

## **Print Leaf**

In [81]:
def print_leaf(counts):
    """A nicer way to print the predictions at a leaf."""
    total = sum(counts.values()) * 1.0
    probs = {}
    for lbl in counts.keys():
        probs[lbl] = str(int(counts[lbl] / total * 100)) + "%"
    return probs

In [82]:
# Printing that a bit nicer
print_leaf(classify(training_data[0], my_tree))

{'Grape': '100%'}

In [83]:
# On the second example, the confidence is lower
print_leaf(classify(training_data[1], my_tree))

{'Lemon': '100%'}

In [84]:
# Evaluate
testing_data = [
    ['Green', 3, 'Apple'],
    ['Yellow', 4, 'Apple'],
    ['Red', 2, 'Grape'],
    ['Red', 1, 'Grape'],
    ['Yellow', 3, 'Lemon'],
]

In [85]:
for row in testing_data:
    print ("Actual: %s. Predicted: %s" %
           (row[-1], print_leaf(classify(row, my_tree))))

Actual: Apple. Predicted: {'Grape': '100%'}
Actual: Apple. Predicted: {'Lemon': '100%'}
Actual: Grape. Predicted: {'Apple': '100%'}
Actual: Grape. Predicted: {'Apple': '100%'}
Actual: Lemon. Predicted: {'Lemon': '100%'}
